# Arrival-time distributions from saved IBM Quantum jobs

Loads downloaded IBM Quantum Sampler jobs (zip files containing `job-<id>-result.json` and `job-<id>-info.json`), applies the U(1) post-selection and plots

1. the cumulative arrival probability $P_\Sigma(t_n)$, together with the ideal (noiseless) prediction,
2. the local magnetization map,
3. the error distribution (particle-number sectors before post-selection).

**File names.** The system is read from the file name, e.g. `Zeno_1d_L7_dt1.zip` → 1D chain, $L=7$, $\Delta t=1$; `Zeno_2d_L4_dt2.zip` → $4\times4$ lattice, $\Delta t=2$ (`L4x3` for a $4\times3$ lattice; `dt0.5` or `dt0p5` for $\Delta t=0.5$). The number of up-spins $k$ is detected from the data (most common particle number at the first time), or can be set by hand.

**Circuit layout assumed** (as in the experiment code): pub $i$ is the circuit ending at $t_n=(i+1)\Delta t$; classical bits $0\ldots N-1$ hold the final measurement of all qubits, the following bits the detector records in chronological order; the detector is the last qubit. No IBM account is needed.

## 1. Imports

In [1]:
import glob, json, os, re, zipfile
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_ibm_runtime import RuntimeDecoder     # decodes the saved result files

## 2. Parameters — change these

In [ ]:
JOB_FILES = sorted(glob.glob("Zeno_*.zip"))   # all jobs in this folder; or e.g.
# JOB_FILES = sorted(glob.glob("Zeno_1d_L7_dt*.zip"))      # one system, all Delta t
# JOB_FILES = ["data/Zeno_1d_L7_dt1.zip", "data/Zeno_1d_L7_dt3.zip"]

K_OVERRIDE = None        # None: detect the number of up-spins from the data; or an integer

# circuit parameters of the experiment (used only for the ideal reference curves)
DELTA = 0.5              # anisotropy of the XXZ gate exp[-i tau (XX + YY + Delta ZZ)]
TAU = 0.25               # gate parameter tau
LAYER_TIME = {1: 0.5, 2: 0.25}   # time per brickwork layer in 1D and 2D
SHOW_IDEAL = True        # add the noiseless prediction as dashed lines

# which job is shown in the magnetization map and the error distribution (index in JOB_FILES)
MAP_JOB = 0
N_SNAPSHOTS = 12         # 2D only: number of lattice snapshots

SAVE_NPZ = False         # True: save the extracted arrays as <zip name>.npz next to the zip
print("\n".join(JOB_FILES) if JOB_FILES else "No files found - check JOB_FILES")

No files found - check JOB_FILES


## 3. Loading the jobs

In [3]:
def parse_name(path):
    """System parameters from a file name such as Zeno_1d_L7_dt1.zip or Zeno_2d_L4x4_dt0p5.zip."""
    name = os.path.basename(path)
    m = re.search(r"(\d)d_L(\d+)(?:x(\d+))?_dt([0-9.p]+?)(?:\.zip|_|$)", name, re.IGNORECASE)
    if m is None:
        raise ValueError(f"cannot read dimension, L and dt from '{name}'")
    dim = int(m.group(1))
    Lx = int(m.group(2))
    Ly = int(m.group(3)) if m.group(3) else (Lx if dim == 2 else 1)
    Dt = float(m.group(4).replace("p", "."))
    return dict(name=name, dim=dim, Lx=Lx, Ly=Ly, n_qubits=Lx * Ly, Dt=Dt)


def load_job(path):
    """Read one downloaded job: returns (list of counts dicts, one per pub; job info dict)."""
    with zipfile.ZipFile(path) as z:
        files = z.namelist()
        res_file = next(f for f in files if f.endswith("-result.json"))
        info_file = next((f for f in files if f.endswith("-info.json")), None)
        result = json.loads(z.read(res_file), cls=RuntimeDecoder)      # PrimitiveResult
        info = json.loads(z.read(info_file)) if info_file else {}
    counts = []
    for pub in result:
        data = pub.data
        reg = getattr(data, "cr", None)                                # register used in the experiment
        if reg is None:
            reg = getattr(data, list(data.keys())[0])
        counts.append(reg.get_counts())
    return counts, info

## 4. Post-processing
Same analysis as in the experiment: U(1) post-selection, first-detection probability $P_a(t_n)$, occupations and particle-number distribution.

In [4]:
def bits_of(bitstring, n_bits):
    """Qiskit bitstrings are little-endian: classical bit 0 is the right-most character."""
    s = bitstring.replace(" ", "")
    return np.array([int(s[-1 - i]) for i in range(n_bits)], dtype=np.int64)


def analyse_counts(counts, n_qubits, n_meas, k, detector):
    """P_a, kept shots, occupations and particle-number distribution at one time t_n."""
    n_bits = n_qubits + n_meas - 1
    weight = np.zeros(n_qubits + 1)
    occ = np.zeros(n_qubits)
    N_kept = n_first = 0
    for bitstring, c in counts.items():
        b = bits_of(bitstring, n_bits)
        final, records = b[:n_qubits], b[n_qubits:]
        w = int(final.sum())
        weight[w] += c
        if w != k:                     # post-selection on the particle number
            continue
        N_kept += c
        occ += c * final
        if records.sum() == 0 and final[detector] == 1:   # first detection at t_n
            n_first += c
    total = sum(counts.values())
    return dict(P_a=n_first / N_kept if N_kept else np.nan, N_kept=N_kept,
                occ=occ / N_kept if N_kept else np.full(n_qubits, np.nan),
                weight=weight / total, shots=total)


def detect_k(counts0, n_qubits):
    """Number of up-spins: most frequent particle number in the first circuit."""
    w = np.zeros(n_qubits + 1)
    for bitstring, c in counts0.items():
        w[bits_of(bitstring, n_qubits).sum()] += c
    return int(np.argmax(w))


def process_job(path):
    sysp = parse_name(path)
    counts, info = load_job(path)
    N = sysp["n_qubits"]
    k = K_OVERRIDE if K_OVERRIDE is not None else detect_k(counts[0], N)
    out = [analyse_counts(c, N, n_meas=i + 1, k=k, detector=N - 1) for i, c in enumerate(counts)]
    P_a = np.array([o["P_a"] for o in out])
    N_kept = np.array([o["N_kept"] for o in out])
    var_a = P_a * (1 - P_a) / np.maximum(N_kept, 1)          # Eq. (B1)
    r = dict(sysp, k=k, backend=info.get("backend", "?"), job_id=info.get("id", "?"),
             t=sysp["Dt"] * np.arange(1, len(counts) + 1), P_a=P_a,
             P_sigma=np.cumsum(P_a), sigma=np.sqrt(np.cumsum(var_a)),   # Eqs. (7), (B2)
             occ=np.array([o["occ"] for o in out]),
             weight=np.array([o["weight"] for o in out]),
             kept_fraction=N_kept / np.array([o["shots"] for o in out]))
    if SAVE_NPZ:
        np.savez(os.path.splitext(path)[0] + ".npz",
                 **{key: v for key, v in r.items() if isinstance(v, np.ndarray)})
    return r


jobs = [process_job(f) for f in JOB_FILES]
for r in jobs:
    size = f"L = {r['Lx']}" if r["dim"] == 1 else f"{r['Lx']} x {r['Ly']}"
    print(f"{r['name']}: {r['backend']}, job {r['job_id']}, {r['dim']}D, {size}, "
          f"k = {r['k']}, Dt = {r['Dt']:g}, {len(r['t'])} times, "
          f"P_Sigma(t_max) = {r['P_sigma'][-1]:.3f} +- {r['sigma'][-1]:.3f}")

## 5. Ideal reference
Exact noiseless first-detection probabilities for the same circuit, computed with a statevector: after every monitoring period the probability of a click is recorded and the detector is projected onto 'down'.

In [5]:
def brickwork_layers(dim, Lx, Ly):
    """Bond layers of the experiment: 1D even/odd; 2D even-x, odd-x, even-y, odd-y."""
    if dim == 1:
        L = Lx * Ly
        return [[(i, i + 1) for i in range(0, L - 1, 2)],
                [(i, i + 1) for i in range(1, L - 1, 2)]]
    s = lambda x, y: y * Lx + x
    return [[(s(x, y), s(x + 1, y)) for y in range(Ly) for x in range(0, Lx - 1, 2)],
            [(s(x, y), s(x + 1, y)) for y in range(Ly) for x in range(1, Lx - 1, 2)],
            [(s(x, y), s(x, y + 1)) for x in range(Lx) for y in range(0, Ly - 1, 2)],
            [(s(x, y), s(x, y + 1)) for x in range(Lx) for y in range(1, Ly - 1, 2)]]


def ideal_first_detection(r):
    """Noiseless P_a(t_n) at the times of job r."""
    N, dim = r["n_qubits"], r["dim"]
    lpp = int(round(r["Dt"] / LAYER_TIME[dim]))              # layers per monitoring period
    layers = []
    for bonds in brickwork_layers(dim, r["Lx"], r["Ly"]):
        qc = QuantumCircuit(N)
        for i, j in bonds:                                     # exp[-i tau (XX+YY+Delta ZZ)]
            qc.rxx(2 * TAU, i, j); qc.ryy(2 * TAU, i, j); qc.rzz(2 * TAU * DELTA, i, j)
        layers.append(qc)
    psi = np.zeros(2 ** N, complex); psi[sum(1 << q for q in range(r["k"]))] = 1.0
    psi = Statevector(psi)
    up = ((np.arange(2 ** N) >> (N - 1)) & 1).astype(bool)    # detector up
    P, ell = [], 0
    for _ in range(len(r["t"])):
        for _ in range(lpp):
            psi = psi.evolve(layers[ell % len(layers)]); ell += 1
        v = psi.data.copy()
        P.append(np.sum(np.abs(v[up]) ** 2))
        v[up] = 0.0                                            # no click: project
        psi = Statevector(v)
    return np.array(P)


if SHOW_IDEAL:
    for r in jobs:
        r["P_a_ideal"] = ideal_first_detection(r)
        r["P_sigma_ideal"] = np.cumsum(r["P_a_ideal"])

## 6. Cumulative arrival probability $P_\Sigma(t_n)$
Solid: hardware after post-selection, band = one standard deviation. Dashed: ideal prediction.

In [6]:
groups = {}                                    # one panel per system (dim, size, k)
for r in jobs:
    groups.setdefault((r["dim"], r["Lx"], r["Ly"], r["k"]), []).append(r)

fig, axes = plt.subplots(1, len(groups), figsize=(5.5 * len(groups), 4.0), squeeze=False)
for ax, (key, rs) in zip(axes[0], groups.items()):
    rs = sorted(rs, key=lambda r: r["Dt"])
    colors = plt.cm.viridis(np.linspace(0.0, 0.85, len(rs)))
    for c, r in zip(colors, rs):
        t = np.concatenate(([0.0], r["t"]))
        P = np.concatenate(([0.0], r["P_sigma"]))
        s = np.concatenate(([0.0], r["sigma"]))
        ax.plot(t, P, "-o", ms=3, color=c, label=rf"$\Delta t={r['Dt']:g}$")
        ax.fill_between(t, P - s, P + s, color=c, alpha=0.2, lw=0)
        if SHOW_IDEAL:
            ax.plot(t, np.concatenate(([0.0], r["P_sigma_ideal"])), "--", color=c)
    dim, Lx, Ly, k = key
    size = f"L={Lx}" if dim == 1 else f"{Lx}x{Ly}"
    ax.set_title(f"{dim}D, {size}, k={k} ({rs[0]['backend']})")
    ax.set_xlabel(r"Time, $t$")
    ax.set_ylabel(r"Cumulative arrival probability, $P_\Sigma(t_n)$")
    ax.set_ylim(0, 1.02)
    if SHOW_IDEAL:
        ax.plot([], [], "--", color="0.4", label="ideal")
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

ValueError: Number of columns must be a positive integer, not 0

<Figure size 0x400 with 0 Axes>

## 7. Local magnetization map
Up-spin probability of every qubit at the monitoring times (post-selected, averaged over all measurement records), for job `MAP_JOB`.

In [ ]:
r = jobs[MAP_JOB]
if r["dim"] == 1:
    fig, ax = plt.subplots(figsize=(4.8, 4.6))
    im = ax.pcolormesh(np.arange(1, r["n_qubits"] + 2) - 0.5, np.concatenate(([0.0], r["t"])),
                       r["occ"], cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(1, r["n_qubits"] + 1))
    ax.set_xlabel("Qubit index"); ax.set_ylabel(r"Time, $t$")
    ax.set_title(rf"{r['name']}: local magnetization")
    fig.colorbar(im, ax=ax, label=r"Up-spin probability $\langle n_j(t)\rangle$")
    plt.tight_layout()
else:
    n_show = min(N_SNAPSHOTS, len(r["t"]))
    ncol = min(6, n_show); nrow = int(np.ceil(n_show / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(1.9 * ncol, 2.0 * nrow), squeeze=False)
    for a in axes.flat:
        a.axis("off")
    for m in range(n_show):
        a = axes.flat[m]
        im = a.imshow(r["occ"][m].reshape(r["Ly"], r["Lx"]), cmap="Blues", vmin=0, vmax=1,
                      origin="lower")
        a.set_title(rf"$t={r['t'][m]:g}$", fontsize=10)
        a.axis("on"); a.set_xticks([]); a.set_yticks([])
    fig.suptitle(f"{r['name']}: local magnetization (start bottom-left, detector top-right)",
                 fontsize=11)
    fig.colorbar(im, ax=axes, label=r"$\langle n_j(t)\rangle$", shrink=0.8)
plt.show()

## 8. Error distribution
Fraction of all shots in each particle-number sector before post-selection, for job `MAP_JOB`. Only the sector with $k$ particles (between the dotted lines) is kept.

In [ ]:
r = jobs[MAP_JOB]
fig, ax = plt.subplots(figsize=(5.2, 4.2))
im = ax.pcolormesh(np.arange(r["n_qubits"] + 2) - 0.5, np.concatenate(([0.0], r["t"])),
                   r["weight"], cmap="Blues_r", vmin=0, vmax=1)
for x in (r["k"] - 0.5, r["k"] + 0.5):
    ax.axvline(x, color="w", lw=0.8, ls=":")
ax.set_xticks(range(r["n_qubits"] + 1))
ax.set_xlabel("Total particle number"); ax.set_ylabel(r"Time, $t$")
ax.set_title(f"{r['name']}: error distribution")
fig.colorbar(im, ax=ax, label="Fraction of shots")
plt.tight_layout()
plt.show()
print("fraction kept by post-selection, N_i / N_shot:", np.round(r["kept_fraction"], 3))